# 📊 Online Retail II — Exploratory Data Analysis

**Loyiha:** Multi-Agent Dynamic Pricing System  
**Muallif:** TDIU Data Science  
**Sana:** 2026

Bu notebook Online Retail II datasetining chuqur tahlilini taqdim etadi:
1. Importlar va konfiguratsiya
2. Dataset yuklash
3. Tozalash va preprocessing
4. Tavsifiy statistika
5. Narx taqsimoti vizualizatsiya
6. RFM segmentatsiya
7. Sodiqlik metrikalari
8. Korrelyatsiya tahlili
9. Narx elastikligi

## 1️⃣ Importlar va Konfiguratsiya

In [ ]:
import sys
from pathlib import Path

# Loyiha rootini path ga qo'shish
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import DataLoader
from src.utils import get_color_palette

# Grafik sozlamalari
plt.style.use('dark_background')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.facecolor'] = '#1a2035'
plt.rcParams['figure.facecolor'] = '#0e1117'

COLORS = get_color_palette()
print('Sozlamalar yuklandi.')

## 2️⃣ Dataset Yuklash

Sample data ishlatamiz (Online Retail II formatida). Real dataset uchun:  
https://archive.ics.uci.edu/dataset/502/online+retail+ii

In [ ]:
loader = DataLoader()
df_raw = loader.load_data(PROJECT_ROOT / 'data' / 'sample_data.csv')

print(f'Qatorlar soni: {len(df_raw):,}')
print(f'Ustunlar: {df_raw.columns.tolist()}')
df_raw.head()

## 3️⃣ Tozalash va Preprocessing

In [ ]:
df_clean = loader.clean_data(df_raw)
stats = loader.cleaning_stats

print('Tozalash statistikasi:')
for key, value in stats.items():
    print(f'  {key}: {value:,}')
print(f'\nSaqlanish darajasi: {stats["retention_pct"]}%')

## 4️⃣ Tavsifiy Statistika

In [ ]:
summary = loader.get_summary_stats(df_clean)
summary_df = pd.DataFrame(list(summary.items()), columns=['Metrika', 'Qiymat'])
summary_df

## 5️⃣ Narx Taqsimoti Vizualizatsiya

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram + KDE
axes[0].hist(df_clean['Price'], bins=40, color=COLORS['PPO'], alpha=0.7, edgecolor='white')
axes[0].set_xlabel('Narx (£)', color='white')
axes[0].set_ylabel('Chastota', color='white')
axes[0].set_title('Narxlar Histogrami', color='white', fontsize=14, fontweight='bold')
axes[0].tick_params(colors='white')
axes[0].grid(alpha=0.2)

# Box plot
axes[1].boxplot(df_clean['Price'], vert=False, patch_artist=True,
                boxprops=dict(facecolor=COLORS['secondary'], alpha=0.5))
axes[1].set_xlabel('Narx (£)', color='white')
axes[1].set_title('Narxlar Box Plot', color='white', fontsize=14, fontweight='bold')
axes[1].tick_params(colors='white')
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

## 6️⃣ RFM Hisoblash va Segmentatsiya

In [ ]:
rfm = loader.calculate_rfm(df_clean)
print(f'Jami mijozlar: {len(rfm):,}')
rfm.head()

In [ ]:
# Segment taqsimoti
seg_counts = rfm['Segment'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
seg_colors = {'Champion': '#22c55e', 'Loyal': '#00d4aa', 'Potential': '#6366f1',
              'At_Risk': '#f59e0b', 'Lost': '#ef4444'}
colors_pie = [seg_colors.get(s, '#94a3b8') for s in seg_counts.index]
axes[0].pie(seg_counts.values, labels=seg_counts.index, colors=colors_pie,
            autopct='%1.1f%%', startangle=90, textprops={'color': 'white'})
axes[0].set_title('Segment Taqsimoti', color='white', fontsize=14, fontweight='bold')

# Bar — har segment uchun o'rtacha Monetary
seg_money = rfm.groupby('Segment')['Monetary'].mean().sort_values(ascending=False)
axes[1].bar(seg_money.index, seg_money.values,
            color=[seg_colors.get(s, '#94a3b8') for s in seg_money.index])
axes[1].set_xlabel('Segment', color='white')
axes[1].set_ylabel('O\'rtacha Monetary (£)', color='white')
axes[1].set_title('Segment bo\'yicha O\'rtacha Monetary', color='white', fontsize=14, fontweight='bold')
axes[1].tick_params(colors='white')
axes[1].grid(alpha=0.2, axis='y')

plt.tight_layout()
plt.show()

## 7️⃣ Sodiqlik Metrikalari

RFM_Score taqsimoti va segmentlar orasidagi farqlar.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(rfm['RFM_Score'], bins=range(3, 17), color=COLORS['PPO'],
        alpha=0.8, edgecolor='white')
ax.set_xlabel('RFM Score (3-15)', color='white')
ax.set_ylabel('Mijozlar soni', color='white')
ax.set_title('RFM Score Taqsimoti', color='white', fontsize=14, fontweight='bold')
ax.tick_params(colors='white')
ax.grid(alpha=0.2, axis='y')
plt.show()

## 8️⃣ Korrelyatsiya Tahlili

In [ ]:
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
corr = df_clean[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            cbar_kws={'label': 'Korrelyatsiya'}, ax=ax)
ax.set_title('Korrelyatsiya Matritsasi', color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9️⃣ Narx Elastikligi

OLS regression yordamida elastiklik koeffitsientini hisoblaymiz:

$$\ln(Q) = \alpha + \varepsilon \cdot \ln(P)$$

Bu yerda $\varepsilon$ — narx elastikligi (manfiy bo'lishi kutiladi).

In [ ]:
elast = loader.calculate_elasticity(df_clean)
print(f'Elastiklik (epsilon): {elast["elasticity"]:.4f}')
print(f'Intercept: {elast["intercept"]:.4f}')
print(f'R-squared: {elast["r_squared"]:.4f}')
print(f'Kuzatuvlar soni: {elast["n_obs"]:,}')
print()
interp = 'Elastik' if abs(elast['elasticity']) > 1 else 'Noelastik'
print(f'Talqin: {interp} (|epsilon| {">>" if abs(elast["elasticity"]) > 1 else "<"} 1)')

In [ ]:
# Scatter + regression line
sample = df_clean[(df_clean['Price'] > 0) & (df_clean['Quantity'] > 0)].sample(
    min(2000, len(df_clean)), random_state=42)
log_price = np.log(sample['Price'])
log_qty = np.log(sample['Quantity'])

fig, ax = plt.subplots(figsize=(12, 6))
ax.scatter(log_price, log_qty, alpha=0.4, color=COLORS['PPO'], s=15)
x_line = np.linspace(log_price.min(), log_price.max(), 100)
y_line = elast['intercept'] + elast['elasticity'] * x_line
ax.plot(x_line, y_line, color=COLORS['accent'], linewidth=3,
        label=f'OLS: epsilon = {elast["elasticity"]:.3f}')
ax.set_xlabel('ln(Price)', color='white')
ax.set_ylabel('ln(Quantity)', color='white')
ax.set_title('Log-Log Narx-Talab Regression', color='white', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.tick_params(colors='white')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## ✅ Yakuniy Xulosa

EDA tahlili ko'rsatdiki:
1. **Narxlar** logaritmik-normal taqsimotga yaqin
2. **RFM segmentatsiya** mijozlarni 5 ta strategik guruhga ajratadi
3. **Narx elastikligi** ~ -1.5 (elastik) — narx pasayishi talabni proportional ravishda oshiradi
4. **Korrelyatsiya** Quantity va Price o'rtasida manfiy (kutilgan)

Bu natijalar **Multi-Agent RL muhitini** kalibratsiya qilish uchun asos bo'ladi:
- `elasticity = -1.5` (environment.py da)
- `base_demand = 100`
- `initial_price = 10`